# 05 — Classify the OOM Without Summoning One

**Target:** Apache Spark 4.2.0; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
Can we collect evidence to distinguish JVM heap pressure, native thread limits, Python worker memory and a kernel kill without causing an OOM?

**Why this matters:** the label “OOM” hides failures with different owners and remedies.

In [ ]:
from pyspark.sql import SparkSession
import os, pathlib, subprocess, sys
spark=SparkSession.builder.master('local[2]').appName('thread-lab-05').getOrCreate()
print('Spark',spark.version,'Python',sys.version.split()[0],'driver PID',os.getpid())
for key in ['spark.executor.memory','spark.executor.memoryOverhead','spark.executor.pyspark.memory','spark.python.worker.memory']:
    print(key, spark.sparkContext.getConf().get(key,'<unset>'))

In [ ]:
# Linux cgroup v2 counters, if present. Missing files are normal on other OSes.
for name in ['memory.current','memory.max','memory.events','pids.current','pids.max','pids.events']:
    path=pathlib.Path('/sys/fs/cgroup')/name
    print(f'{name}:',path.read_text().strip() if path.exists() else '<not available here>')

## Evidence matrix
- `Java heap space`: inspect GC logs and heap dump/dominators.
- `unable to create native thread`: inspect JVM thread dump, OS/cgroup PID limits and native resources.
- `OOMKilled`: correlate cgroup `memory.events`, orchestrator events and process/container memory.
- Python worker exit: correlate worker logs/process RSS and executor container budget.
- Direct buffer failure: inspect direct-buffer use and exact JVM message.

**Expected:** this notebook reports available configuration/cgroup data only; it does not prove a cause.

**Why:** NMT must be enabled before the JVM starts and does not count separate Python processes. `memoryOverhead` is not a heap partition; PySpark memory limiting is platform dependent. Avoid `-Xmx × thread count` style arithmetic as a measure of resident RAM.

In [ ]:
spark.stop()